In [1]:
import random
from pathlib import Path
from main import load_dotenv
from data_loader.raw_dataloader import RawDataloader
from models.ngram.knn import KNN
from models.ngram.model import Model
from collections import Counter, defaultdict

load_dotenv(Path("../.env"))

/home/bugslayer/Documents/research/MusicGeneration/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataloader = RawDataloader()
dataloader.prepare_dataset(representation="note_table")
loader = dataloader.loader(load_music=dataloader.load_note_table_music)

In [3]:
tokens_by_pitch = defaultdict(Counter)

for batch in loader:
    for song in batch:
        note_tokens = zip(
            song["pitch"],
            song["velocity_bin"],
            song["delta_onset_bin"],
            song["duration_bin"],
        )

        for token in note_tokens:
            tokens_by_pitch[token[0]][token] += 1

knn = KNN(n=10, min_count=10)
neighbors = knn.get_nearest_neighbors(tokens_by_pitch)

In [4]:
ngram_model = Model(loader=loader, neighbors=neighbors, n=3)
ngram_model.train()

In [28]:
BOS = ("<BOS>", "<BOS>", "<BOS>", "<BOS>")
EOS = ("<EOF>", "<EOF>", "<EOF>", "<EOF>")

In [29]:
next_token = None
tokens = (BOS, BOS)

while next_token != EOS:
    next_token = ngram_model.predict(tokens)[0]
    tokens += (next_token,)

In [31]:
len(tokens)

11406

In [25]:
len(batch[0]["pitch"])

6625